# RAG

Source: https://codelabs.developers.google.com/codelabs/production-ready-ai-with-gc/8-advanced-rag-methods/advanced-rag-methods#0

Retrieval Augmented Generation (RAG) "migliora" le prestazioni degli LLM fondandole su conoscenze esterne. In poche parole, questa tecnica consente alle LLM di poter accedere a delle informazioni di cui altrimenti non avrebbe visibilità e/o accesso. Per esempio, supponiamo di essere un azienda che ha tanta documentazione aziendale. Questa documentazione potrebbe essere sensibile o in continuo aggiornamento. Una qualsiasi LLM non ne avrebbe accesso diretto. La RAG consente all'LLM di potervi accedervi e consultarne i contenuti. 

E' quindi un fine-tuning? No. Non viene addestrato alcun modello. Non vengono aggiornati i pesi della LLM. Lei rimane tale. Inoltre, come detto prima, se il dato è in costante aggiornamento non sarebbe nemmeno sostenibile fare il fine tuning di una LLM ogni volta.

La RAG solitamente si divide nei seguenti steps:

- Indexing: è la parte fatta preliminarmente. Non viene eseguita ogni volta che viene chiamata la LLM. In questa parte la documentazione a disposizione viene transformata in **embeddings** e caricata su un **database vettoriale**. Esistono differenti modi con il quale fare questa cosa e una delle cose più importanti a riguardo è la suddivisione in **chunk** dei documenti. Non è possibile infatti creare un unico embedding per un singolo documento. Non sarebbe sostenibile e si perderebbero molte informazioni. Da sottolineare inoltre che esistono molte tipologie di database vettoriali, es **PostgreSQL**, **ChromaDB**, ect 
- Retrieval: è la parte con il quale vengono recuperati dal database gli embedding più similari alla domanda che ha fatto l'utente. Ovvero va a recuperare dalla documentazione l'informazione per poter rispondere alla domanda. Anche questa parte non è banale. Molto spesso i retrieval non sono soddisfacenti e quindi è opportuno integrare tecniche di **reranking**.
- Generative: in questa parte finalmente la LLM avrà a disposizione il testo dei K chunck più similari e può di conseguenza generare la sua risposta in linguaggio naturale.

## Domande:

1. Come posso gestire l'aggiornamento in tempo reale del database vettoriale? Ovvero aggiornare il DB nel momento in cui gli utenti stanno usando attivamente la RAG?

    - **Pipeline di Sincronizzazione Periodica (Cron Job / ETL)**: Per documenti che cambiano con una frequenza accettabile (es. ogni notte o ogni ora), si possono pianificare dei job di sincronizzazione (usando strumenti come Airflow o script schedulati). Il job scansiona la sorgente, confronta i timestamp o gli hash dei file e aggiorna nel vettore solo ciò che è effettivamente cambiato. Approccio adottato da me nell'esperienza in Abstract.

In [1]:
# Ci serve quindi un modello di embedding: ovvero un modello che dato un testo lo converta in embeddings vettoriali. Ne esistono sia a pagamento che non.
# E non per forza serve usare langchain per farlo.

from langchain_ollama import OllamaEmbeddings

# Questo per esempio è un modello open-source: non si paga.
embeddings = OllamaEmbeddings(model="bge-m3", validate_model_on_init=True)

# Example.
sentences = ["This is an example sentence", "Each sentence is converted"]

embedded_sentences = embeddings.embed_documents(sentences)
len(embedded_sentences), len(embedded_sentences[0])

(2, 1024)

Come si può vedere esistono diversi modelli con cui fare embedding. Più complessi, meno complessi. Opne-source o a pagamento. Da scegliere in base alle esigenze.

Da notare come langchain è una libreria per l'orchestrazione e per lo sviluppo di applicazioni basate sulle LLM che in questo caso ha creato internamente delle interfacce atte ad uniformare l'output di vari modelli provenienti da diversi providers.

## Indexing

Per prima cosa abbiamo detto come dividiamo i documenti? Ci sono diverse tecniche.

- **Character Splitter**: separa il testo basandosi su un singolo separatore fisso (es. "\n").
- **Recursive Splitter**: prova una lista gerarchica di separatori in sequenza: ["\n\n", "\n", " ", ""]. Tenta prima di dividere per paragrafi; se il pezzo è ancora troppo grande, passa alle frasi, poi alle parole e infine ai singoli caratteri.
- **Token Splitter**: converte il testo in token e lo taglia al raggiungimento esatto del numero di token specificato.

Ora serve istanziare un database. In questo caso usiamo PostgreSQL. Per farmo bisogna scaricare PostgreSQL, creare le credenziali e attivare l'estensione pgvector.

Una volta fatto ciò possiamo creare la connessione al nostro database (in questo caso in locale).

In [2]:
import urllib
from langchain_postgres import PGEngine, PGVectorStore
from langchain_postgres.v2.engine import Column

In [3]:
DB_USER = "postgres"
DB_PASSWORD = "Beneke13!!!!"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "chatbot"

encoded_password = urllib.parse.quote_plus(DB_PASSWORD)

connection = f"postgresql://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

pg_engine = PGEngine.from_connection_string(url=connection)

In [4]:
vector_store = PGVectorStore.create_sync(
    engine=pg_engine,
    table_name="ml_vectors",
    embedding_service=embeddings,
    metadata_columns=["source"],
)

In [5]:
# Questo è il retrieval classico dove prendiamo le prime top 10 chunk.
query = "Qual è il periodo di preavviso in caso di dimissioni per un dipendente di 4° livello di 4 anni e mezzo di anzianità aziendale?"
retrieved_docs = vector_store.similarity_search(query, k=5)
retrieved_docs[0]

Document(id='4e898ddd-52ae-47d6-a643-f08b802080e5', metadata={'page': 124, 'source': 'unimatica'}, page_content='buzione, all’impiegato dovrà essere intanto corrisposta la parte di retribuzione non con-\ntestata.\nNOTA A VERBALE\nLe Parti, preso atto del difficile contesto di crisi economica che sta colpendo in modo\nparticolare le PMI del settore, al fine di apportare benefici al sistema impresa nel suo\ncomplesso, convengono di promuovere, su richiesta delle imprese interessate, previa in-\nformativa in tempi congrui alla RSU o, ove assente, alle OOSS territoriali, la sperimen-\ntazione di un modello organizzativo che preveda la possibilità per le imprese stesse di\nerogare le somme relative ai ratei di mensilità aggiuntive, premi e tutti gli elementi eco-\nnomici accessori anziché in un’unica soluzione in dodicesimi.\nArt. 82 - Preavviso di licenziamento e di dimissioni\nIl licenziamento dell’impiegato non in prova e non ai sensi dell’Art. 61 – Disciplina del\nlavoro - o le sue dimi

Esistono anche tecniche di **hybrid search** in cui vengono combinati gli approcci densi e quelli sparsi. Spiega la combinazione di Dense Retrieval (embeddings vettoriali per la semantica) e Sparse Retrieval (BM25 o Postgres Full-Text Search per le parole chiave esatte).

Avevo visto qualcosa nella mia esperienza in Abstract, non tanto per la parte RAG, ma per la parte di identificazione venditore. Infatti quello che avevo adottato era solo la ricerca sparsa: avevo convertito i testi in embedding con tf-idf che sostanzialmente nient'altro è che un counter avanzato dei caratteri/token che tu decidi di selezionare con un pattern regex all'interno del testo. Non avevo aggiunto la parte semantica mediante modello di embedding più strutturato.

Avevo però all'epoca trovato questo link: https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/about-hybrid-search

## Generativa

In [6]:
# Proviamo ora a vedere la risposta che darebbe una LLM se gli fornissimo questi risultati.

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    project="foini-506814",
    location="europe-west1",
    vertexai=True,
    temperature=0.9,
    max_tokens=1000,
    max_retries=1,
    timeout=120,
    include_thoughts=False,
)

system_prompt= PromptTemplate.from_template(
    template="""
    Sei un assistente utile. Rispondi alla domanda dell'utente basandoti ESCLUSIVAMENTE sul contesto fornito qui sotto. Se l'informazione non è presente nel contesto, dichiara di non saper rispondere.

    --- CONTESTO ---
    {retrieved_text}
    """,
)

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage

retrieved_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

messages = [
    SystemMessage(content=system_prompt.format(retrieved_text=retrieved_text)),
    HumanMessage(name="Pietro", content=query),
]

ai_msg = llm.invoke(messages)
ai_msg

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content='Mi dispiace, ma non posso rispondere alla tua domanda. Il contesto fornito non specifica il periodo di preavviso per un dipendente di 4° livello con 4 anni e mezzo di anzianità aziendale.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e9b9-c5c3-7102-92a1-3717be0a6ff7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1704, 'output_tokens': 47, 'total_tokens': 1751, 'input_token_details': {'cache_read': 0}})

A questo punto nascono delle domande.

## Domande:

1. Cosa faresti se i risultati del retrieval non sono soddisfacenti?

    - Un buon modo di rispondere per partire è quello di capire se il problema è nella parte di retrieval o nel reranking (se presente).
    - Immagino ci siano tante risposte. Una molto banale è sicuramente quello di cambiare modello di embedding o metodo di chunckerizzazione. Nel caso sopra, non so cosa salverò, ma è stato proprio il cambiamento del modello di reranking a portarmi alla risposta della domanda. O meglio usando il modello potente di embedding di Google ho ottenuto la risposta mentre con il modello open source no.

## Reranking

Il reranking risolve ciò aggiungendo un secondo stage.

- Retriever: recupera più candidati, per esempio 25, in quanto se lo può permettere visto che usa una similarity veloce.
- Reranker: usa modelli specifi (come cross-encoder oppure fare una chiamata LLM proprio) per esaminare ogni singolo documento con la corrispondente query. E' più lento ma più accurato. E in questo caso dai 25 cerchiamo di passare a 3 risultati per esempio.

Quindi nella pratica è una corta di ricerca di trade off tra recall e precision.

In [50]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

reranker_tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
reranker_model = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-v2-m3').to("cpu")
reranker_model.eval();

Loading weights: 100%|██████████| 393/393 [00:04<00:00, 81.04it/s]


In [72]:
# Quindi recupero un numero alto di documenti con il retriver.
retrieved_docs = vector_store.similarity_search(query, k=25)

In [73]:
pairs = [[query, doc.page_content] for doc in retrieved_docs]

with torch.no_grad():
    inputs = reranker_tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512).to("cpu")
    scores = reranker_model(**inputs, return_dict=True).logits.view(-1,).float()

In [80]:
import pandas as pd

df = pd.DataFrame({"text": retrieved_docs, "score": scores.numpy()})
df = df.sort_values('score', ascending=False).reset_index(drop=True)

# Reranka e tieni solo i migliori.
reranked_docs = df["text"].tolist()[:3]

In [81]:
from langchain_core.messages import SystemMessage, HumanMessage

retrieved_text = "\n\n".join([doc.page_content for doc in reranked_docs])

messages = [
    SystemMessage(content=system_prompt.format(retrieved_text=retrieved_text)),
    HumanMessage(name="Pietro", content=query),
]

ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='Per gli impiegati che non hanno superato i 5 anni di servizio, il periodo di preavviso per il 4° livello (gruppo B) è di 1 mese e 15 giorni.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e802-4ff1-7803-9579-ed2752468117-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1209, 'output_tokens': 45, 'total_tokens': 1254, 'input_token_details': {'cache_read': 0}})

## Query Transformation

Raramente la domanda grezza dell'utente è ottimale per la ricerca vettoriale.

- HyDE (Hypothetical Document Embeddings): L'LLM genera prima una risposta ipotetica alla domanda; poi si calcola l'embedding della risposta ipotetica per cercare documenti simili nel vector store.
- Multi-Query Expansion: L'LLM riformula la query iniziale in 3-5 varianti differenti per coprire diverse prospettive di ricerca.

Questa parte al momento non l'ho implementata.

## Evalutation

Un'altra domanda cruciale riguardo l'approccio RAG è certamente la seguente: come si valuta l'accuratezza di una RAG? Che di conseguenza porta con sé la conoscenza di quali metriche sono interessanti da guardare e quali framework ci sono a disposizione a riguardo.

Per farlo cercheremo di utilizzare il framework **RAGAS**.

Source: https://docs.ragas.io/en/stable/howtos/integrations/langchain/#evaluate

Le metriche più comuni sono le seguenti:

- Context Relevance: Quanto sono pertinenti i chunk recuperati rispetto alla query?
- Groundedness / Faithfulness: La risposta dell'LLM si basa esclusivamente sul contesto recuperato senza allucinazioni?
- Answer Relevance: La risposta risponde direttamente alla domanda dell'utente?

In [84]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [85]:
sample_queries = [
    "Qual è il periodo di preavviso in caso di dimissioni per un dipendente di 4° livello di 4 anni e mezzo di anzianità aziendale?",
]

expected_responses = [
    "Il periodo di preavviso è di 1 mese e 15 giorni.",
]

from ragas import EvaluationDataset

dataset = []
for query, reference in zip(sample_queries, expected_responses):
    relevant_docs = retriever.invoke(query)
    retrieved_text = "\n\n".join([doc.page_content for doc in relevant_docs])
    messages = [
        SystemMessage(content=system_prompt.format(retrieved_text=retrieved_text)),
        HumanMessage(name="Pietro", content=query),
    ]
    response = llm.invoke(messages)
    dataset.append(
        {
            "user_input": query,
            "retrieved_contexts": [rdoc.page_content for rdoc in relevant_docs],
            "response": response,
            "reference": reference,
        }
    )

evaluation_dataset = EvaluationDataset.from_list(dataset)

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

evaluator_llm = LangchainLLMWrapper(llm)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
)

result